# 03 — Model Training
### NutriFit-AI

Trains **Linear Regression** (baseline) and **Random Forest Regression**
(proposal's main model) for both targets, exactly as specified — no substitutions.

* `RandomizedSearchCV` over `n_estimators`, `max_depth`, `min_samples_leaf`,
  `min_samples_split`, `max_features`
* scored on `neg_mean_absolute_error` so tuning optimises the same quantity the
  evaluation reports
* 5-fold cross-validation, reported as mean ± standard deviation

---

### ⚠️ Runtime: use **CPU**, not GPU

`scikit-learn` is **CPU-only**. `LinearRegression` and `RandomForestRegressor`
cannot use a GPU — an L4 gives exactly zero speed-up here and burns your compute
units for nothing. This whole notebook runs in **1–3 minutes on the free CPU
runtime**.

`Runtime → Change runtime type → CPU`

(The optional GPU benchmark is in notebook 04, section 6 — that one *does* use
the L4, and it is the only cell in the project that should.)

In [ ]:
# ============================================================
# SETUP - run this first in every notebook
# ============================================================
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/NutriFit-AI")
    if not PROJECT.exists():
        raise FileNotFoundError(
            f"{PROJECT} not found.\n"
            "Upload the whole NutriFit-AI folder to the ROOT of your Google Drive "
            "(My Drive/NutriFit-AI), then re-run this cell."
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "scikit-learn>=1.4", "pandas>=2.1", "joblib>=1.3", "seaborn>=0.13"],
        check=False,
    )
else:
    PROJECT = Path.cwd()
    while not (PROJECT / "ml" / "nutrifit").exists() and PROJECT != PROJECT.parent:
        PROJECT = PROJECT.parent

sys.path.insert(0, str(PROJECT / "ml"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nutrifit
from nutrifit import config, data, foods, labels, nutrition, planner, preprocessing, recommender, training

for directory in (config.PROCESSED_DIR, config.ARTIFACTS_DIR, config.FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

print(f"nutrifit  {nutrifit.__version__}")
print(f"project   {PROJECT}")
print(f"data/raw  {config.RAW_DIR}")
print(f"figures   {config.FIGURES_DIR}")
print(f"in colab  {IN_COLAB}")

In [ ]:
def savefig(name):
    """Save the current figure into reports/figures/ for the dissertation."""
    path = config.FIGURES_DIR / f"{name}.png"
    plt.savefig(path)
    print(f"saved {path}")

## 1. Load the labelled dataset

In [ ]:
import logging, time
logging.basicConfig(level=logging.INFO, format="%(levelname)-8s %(message)s")

labelled = pd.read_csv(config.USERS_PROCESSED)
print(f"{len(labelled)} rows, {len(preprocessing.FEATURE_COLUMNS)} features")
print(f"CPU cores available: {os.cpu_count()}")
labelled[["age","gender","weight_kg","bmi","body_fat_pct","fitness_goal",
          "activity_level","calorie_target","protein_target"]].head()

## 2. Train — calorie target

In [ ]:
started = time.perf_counter()
calorie_bundle = training.train_target(
    labelled, "calorie_target", tune=True, n_iter=40, seed=config.RANDOM_SEED
)
print(f"\nTotal wall time: {time.perf_counter() - started:.1f} s\n")

for name, result in calorie_bundle["results"].items():
    m = result.test_metrics
    print(f"{name:20s} MAE={m['mae']:8.2f}  RMSE={m['rmse']:8.2f}  R2={m['r2']:.4f}  "
          f"CV R2={result.cv_metrics['r2']['mean']:.4f} +/- {result.cv_metrics['r2']['std']:.4f}")

print(f"\nBest RF params: {calorie_bundle['results']['RandomForest'].best_params}")

## 3. Train — protein target

In [ ]:
started = time.perf_counter()
protein_bundle = training.train_target(
    labelled, "protein_target", tune=True, n_iter=40, seed=config.RANDOM_SEED
)
print(f"\nTotal wall time: {time.perf_counter() - started:.1f} s\n")

for name, result in protein_bundle["results"].items():
    m = result.test_metrics
    print(f"{name:20s} MAE={m['mae']:8.2f}  RMSE={m['rmse']:8.2f}  R2={m['r2']:.4f}  "
          f"CV R2={result.cv_metrics['r2']['mean']:.4f} +/- {result.cv_metrics['r2']['std']:.4f}")

print(f"\nBest RF params: {protein_bundle['results']['RandomForest'].best_params}")

## 4. Comparison table (goes straight into the report)

In [ ]:
comparison = training.comparison_table([calorie_bundle, protein_bundle])
display(comparison)
comparison.to_csv(config.REPORTS_DIR / "model_comparison.csv", index=False)
print(f"saved {config.REPORTS_DIR / 'model_comparison.csv'}")

## 5. Per-fold cross-validation detail

Reporting mean ± sd across folds — not a single split — is what makes the
evaluation credible.

In [ ]:
rows = []
for bundle in (calorie_bundle, protein_bundle):
    for name, result in bundle["results"].items():
        for metric in ("mae", "rmse", "r2"):
            for fold, value in enumerate(result.cv_metrics[metric]["folds"], start=1):
                rows.append({"target": bundle["target"], "model": name,
                             "metric": metric, "fold": fold, "value": value})
folds_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, target in zip(axes, ("calorie_target", "protein_target")):
    subset = folds_df[(folds_df["target"] == target) & (folds_df["metric"] == "mae")]
    sns.boxplot(data=subset, x="model", y="value", ax=ax)
    sns.stripplot(data=subset, x="model", y="value", color="black", size=6, ax=ax)
    ax.set_title(f"{target} — CV MAE across 5 folds"); ax.set_ylabel("MAE")
plt.tight_layout()
savefig("cv_fold_distribution")
plt.show()

display(folds_df.groupby(["target","model","metric"])["value"].agg(["mean","std"]).round(4))

## 6. Save fitted pipelines for the next notebook

In [ ]:
import joblib
joblib.dump(calorie_bundle, config.ARTIFACTS_DIR / "_calorie_bundle.pkl", compress=3)
joblib.dump(protein_bundle, config.ARTIFACTS_DIR / "_protein_bundle.pkl", compress=3)
print("Saved training bundles to", config.ARTIFACTS_DIR)
print("\nNext: 04_evaluation.ipynb")